## Ekstraksi Fitur

### Import Library dan Inisialisasi MediaPipe

In [2]:
import os
import cv2
import numpy as np
import tensorflow as tf
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import tqdm

print("Menginisialisasi MediaPipe Hand Landmarker...")

# Pastikan file hand_landmarker.task ada di direktori kamu
model_path = 'hasil_data_preparation/hand_landmarker.task'

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

print("✓ MediaPipe Hand Landmarker Berhasil Dinyalakan!")

Menginisialisasi MediaPipe Hand Landmarker...
✓ MediaPipe Hand Landmarker Berhasil Dinyalakan!


I0000 00:00:1780506255.862941   19500 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1780506255.866881   19514 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics (ICL GT1)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780506255.888565   19504 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780506255.905399   19504 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


### Fungsi Ekstraktor Sekuensial

In [7]:
def extract_sequence_robust(base_dir, output_data_path, output_label_path, sequence_length=30, target_sequences_per_class=100):
    X_data = []
    Y_labels = []
    
    classes = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
    class_mapping = {class_name: idx for idx, class_name in enumerate(classes)}
    
    print(f"Menemukan {len(classes)} Kelas Target.")
    
    for class_name in classes:
        class_dir = os.path.join(base_dir, class_name)
        print(f"\n[Proses] Mengekstrak Suku Kata: {class_name}")
        
        image_files = sorted([f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        total_images = len(image_files)
        
        if total_images == 0:
            print(f"❌ Folder {class_name} kosong! Dilewati.")
            continue
            
        # 1. Ekstraksi koordinat gambar yang ada di folder kelas saat ini
        raw_landmarks = []
        for img_name in tqdm.tqdm(image_files, desc=f"MediaPipe {class_name}"):
            img_path = os.path.join(class_dir, img_name)
            frame = cv2.imread(img_path)
            if frame is None:
                continue
                
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image_obj = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            detection_result = detector.detect(mp_image_obj)
            
            if detection_result.hand_landmarks:
                hand_landmarks = detection_result.hand_landmarks[0]
                frame_features = []
                for lm in hand_landmarks:
                    frame_features.extend([lm.x, lm.y, lm.z])
                raw_landmarks.append(frame_features)
            else:
                # Padding jika tangan luput dari kamera
                raw_landmarks.append([0.0] * 63)
        
        if len(raw_landmarks) == 0:
            raw_landmarks = [[0.0] * 63]
            
        # 2. ALGORITMA PENYELAMATAN DATA (OVERSAMPLING SEKUENS DINAMIS)
        generated_sequences = 0
        while generated_sequences < target_sequences_per_class:
            # Jika gambar asli kurang dari 30 frame, kita duplikasi acak frame yang ada sampai genap 30
            if len(raw_landmarks) < sequence_length:
                seq = list(raw_landmarks)
                while len(seq) < sequence_length:
                    seq.append(raw_landmarks[np.random.randint(0, len(raw_landmarks))])
                X_data.append(seq)
                Y_labels.append(class_mapping[class_name])
                generated_sequences += 1
            else:
                # Jika gambarnya banyak, kita ambil jendela 30 frame secara acak berurutan (random slicing)
                max_start_idx = len(raw_landmarks) - sequence_length
                start_idx = 0 if max_start_idx == 0 else np.random.randint(0, max_start_idx)
                
                seq = raw_landmarks[start_idx : start_idx + sequence_length]
                X_data.append(seq)
                Y_labels.append(class_mapping[class_name])
                generated_sequences += 1

    # Memastikan folder tujuan fisik terbentuk
    output_dir = os.path.dirname(output_data_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    # Konversi ke NumPy Array
    X_data = np.array(X_data, dtype=np.float32)
    Y_labels = np.array(Y_labels, dtype=np.int32)
    
    np.save(output_data_path, X_data)
    np.save(output_label_path, Y_labels)
    
    print(f"\n✓ SELESAI! File array berhasil disetarakan secara robust.")
    print(f"-> Dimensi X: {X_data.shape}")
    print(f"-> Dimensi Y: {Y_labels.shape}")
    
    return class_mapping

### Eksekusi Trigger Ekstraksi Dataset

In [9]:
TRAIN_DIR = '3_final_dataset/train'
VAL_DIR = '3_final_dataset/val'

print("--- JALANKAN EKSTRAKSI + PENYEIMBANGAN DATA TRAINING ---")
label_dictionary = extract_sequence_robust(
    base_dir=TRAIN_DIR,
    output_data_path='hasil_data_preparation/X_train_coor.npy',
    output_label_path='hasil_data_preparation/Y_train_coor.npy',
    sequence_length=30,
    target_sequences_per_class=150  # Tiap kelas dilatih dengan kuota seimbang 150 sekuens
)

print("\n--- JALANKAN EKSTRAKSI + PENYEIMBANGAN DATA VALIDASI ---")
extract_sequence_robust(
    base_dir=VAL_DIR,
    output_data_path='hasil_data_preparation/X_val_coor.npy',
    output_label_path='hasil_data_preparation/Y_val_coor.npy',
    sequence_length=30,
    target_sequences_per_class=30   # Tiap kelas divalidasi dengan kuota seimbang 30 sekuens
)

print("\nKamus Pemetaan Akhir:")
print(label_dictionary)

--- JALANKAN EKSTRAKSI + PENYEIMBANGAN DATA TRAINING ---
Menemukan 26 Kelas Target.

[Proses] Mengekstrak Suku Kata: A


MediaPipe A: 100%|██████████| 180/180 [00:03<00:00, 45.98it/s]



[Proses] Mengekstrak Suku Kata: B


MediaPipe B: 100%|██████████| 180/180 [00:05<00:00, 32.42it/s]



[Proses] Mengekstrak Suku Kata: C


...


[Proses] Mengekstrak Suku Kata: Y


MediaPipe Y: 100%|██████████| 20/20 [00:00<00:00, 32.83it/s]



[Proses] Mengekstrak Suku Kata: Z


MediaPipe Z: 100%|██████████| 15/15 [00:00<00:00, 31.27it/s]


✓ SELESAI! File array berhasil disetarakan secara robust.
-> Dimensi X: (780, 30, 63)
-> Dimensi Y: (780,)

Kamus Pemetaan Akhir:
{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25}
